In [ ]:
!pip install -q \
requests \
beautifulsoup4 \
tqdm \
faiss-cpu \
sentence-transformers \
transformers \
accelerate \
torch \
langchain \
langchain-community \
langchain-text-splitters \
pypdf \
readability-lxml

In [ ]:
import os
import requests
import time
import faiss
import numpy as np
import pickle

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from tqdm import tqdm
from readability import Document

from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

SAVE_PATH = "../index/"
PDF_PATH = SAVE_PATH + "/pdfs"

os.makedirs(SAVE_PATH, exist_ok=True)
os.makedirs(PDF_PATH, exist_ok=True)


In [ ]:
import time
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup
import requests

START_URLS = [
    "https://www.iitp.ac.in/",
    "https://academics.iitp.ac.in/",
    "https://www.iitp.ac.in/images/pdf/MTech%20in%20AI-23-11-23.pdf",
    "https://academics.iitp.ac.in/files/mtech_syllabus/mtech-cse.pdf",
    "https://academics.iitp.ac.in/images/tt/Calendar-Spring-2026.pdf",
    "https://drive.google.com/file/d/1rR_Pgaxu3DfGWjlPS5q_Y1vVytnYG0Da/view?usp=sharing"
]

visited = set()
html_pages = []
pdf_links = []

def is_valid(url):
    parsed = urlparse(url)
    return ("iitp.ac.in" in parsed.netloc) and (url not in visited)

def crawl(url, max_pages=20):

    queue = [url]

    while queue and len(visited) < max_pages:

        current = queue.pop(0)

        try:
            # -------- STATUS PRINT --------
            print(
                f"[{len(visited)+1}/{max_pages}] Crawling:",
                current
            )
            print(
                f"Queue: {len(queue)} | HTML: {len(html_pages)} | PDFs: {len(pdf_links)}"
            )

            res = requests.get(current, timeout=10)
            visited.add(current)

            # ---------- PDF ----------
            if ".pdf" in current.lower():
                pdf_links.append(current)
                continue

            # ---------- HTML ----------
            soup = BeautifulSoup(res.text, "html.parser")
            html_pages.append((current, res.text))

            # ---------- Extract Links ----------
            for link in soup.find_all("a", href=True):
                new_url = urljoin(current, link["href"])
                if is_valid(new_url):
                    queue.append(new_url)

            time.sleep(0.3)

        except Exception as e:
            print("Error:", current)
            continue


In [ ]:
for url in START_URLS:
    print("Crawling:", url)
    crawl(url)

print("HTML pages:", len(html_pages))
print("PDFs:", len(pdf_links))

In [ ]:
documents = []

for url, html in tqdm(html_pages):
    try:
        doc = Document(html)
        clean_html = doc.summary()

        soup = BeautifulSoup(clean_html, "html.parser")
        text = soup.get_text(separator="\n")

        documents.append({
            "text": text,
            "source": url
        })
    except:
        continue


In [ ]:
# ---- LOAD MANUALLY ADDED PDFs ----
import os

print("\nLoading manually added PDFs...")

manual_pdfs = os.listdir(PDF_PATH)

for pdf_file in manual_pdfs:
    path = os.path.join(PDF_PATH, pdf_file)

    try:
        loader = PyPDFLoader(path)
        pages = loader.load()

        for p in pages:
            documents.append({
                "text": p.page_content,
                "source": f"manual_pdf:{pdf_file}"
            })

    except Exception as e:
        print("Failed:", pdf_file)


In [ ]:
for pdf_url in tqdm(pdf_links):
    try:
        fname = pdf_url.split("/")[-1]
        path = f"{PDF_PATH}/{fname}"

        r = requests.get(pdf_url)
        with open(path, "wb") as f:
            f.write(r.content)

        loader = PyPDFLoader(path)
        pages = loader.load()

        for p in pages:
            documents.append({
                "text": p.page_content,
                "source": pdf_url
            })

    except:
        continue


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = []
metadata = []

for doc in documents:
    splits = splitter.split_text(doc["text"])
    for s in splits:
        chunks.append(s)
        metadata.append(doc["source"])

print("Total chunks:", len(chunks))


In [ ]:
embed_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5",
    device="cuda"
)

embeddings = embed_model.encode(
    chunks,
    batch_size=64,
    show_progress_bar=True
)


In [ ]:
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings))

faiss.write_index(index, SAVE_PATH + "index.faiss")
np.save(SAVE_PATH + "chunks.npy", chunks)
np.save(SAVE_PATH + "metadata.npy", metadata)

print("✅ IITP RAG index saved successfully")